# CryptoQuant BTC Data Extractor

이 노트북은 CryptoQuant에서 BTC 가격과 미결제 약정(Open Interest) 데이터를 추출합니다.

추출되는 데이터:
- 날짜 (Date)
- BTC 가격 (Price USD)
- 미결제 약정 (Open Interest)

## 1. 필요한 패키지 설치

In [ ]:
# Colab 환경에 필요한 패키지 설치
!pip install selenium pandas webdriver-manager requests -q

## 2. Chrome 브라우저 설정 (Colab용)

In [ ]:
# Colab에서 Chrome과 ChromeDriver 설치
!apt-get update
!apt install -y chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

import sys
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')

## 3. 데이터 추출 코드

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
import json
import pandas as pd
from datetime import datetime

class CryptoQuantScraper:
    def __init__(self, headless=True):
        self.chrome_options = Options()
        if headless:
            self.chrome_options.add_argument('--headless')
        self.chrome_options.add_argument('--no-sandbox')
        self.chrome_options.add_argument('--disable-dev-shm-usage')
        self.chrome_options.add_argument('--disable-gpu')
        self.chrome_options.add_argument('--window-size=1920,1080')
        self.driver = None

    def start_driver(self):
        self.driver = webdriver.Chrome(options=self.chrome_options)

    def extract_highcharts_data(self, url):
        if not self.driver:
            self.start_driver()

        print(f"Loading URL: {url}")
        self.driver.get(url)

        # Wait for chart to load
        print("Waiting for chart to load...")
        time.sleep(8)

        # Extract Highcharts data
        script = """
        try {
            if (typeof Highcharts !== 'undefined' && Highcharts.charts) {
                let allData = [];

                for (let chart of Highcharts.charts) {
                    if (chart && chart.series) {
                        let chartData = { series: [] };

                        chart.series.forEach((series, index) => {
                            if (series && series.data && series.name) {
                                let seriesData = {
                                    name: series.name,
                                    data: series.data.map(point => ({
                                        x: point.x,
                                        y: point.y,
                                        date: new Date(point.x).toISOString()
                                    }))
                                };
                                chartData.series.push(seriesData);
                            }
                        });

                        allData.push(chartData);
                    }
                }

                return JSON.stringify(allData);
            }
            return null;
        } catch (e) {
            return JSON.stringify({error: e.toString()});
        }
        """

        print("Extracting chart data...")
        result = self.driver.execute_script(script)

        if result:
            data = json.loads(result)
            print(f"Extracted data: {len(data)} chart(s) found")
            return self._process_data(data)
        else:
            print("No Highcharts data found")
            return None

    def _process_data(self, raw_data):
        if not raw_data or len(raw_data) == 0:
            return None

        all_records = []

        for chart in raw_data:
            if 'series' not in chart:
                continue

            date_map = {}

            for series in chart['series']:
                series_name = series['name']
                print(f"Processing series: {series_name}")

                for point in series['data']:
                    date = point['date']
                    if date not in date_map:
                        date_map[date] = {
                            'date': date,
                            'timestamp': point['x']
                        }

                    if 'price' in series_name.lower() or 'usd' in series_name.lower():
                        date_map[date]['price_usd'] = point['y']
                    elif 'open interest' in series_name.lower():
                        date_map[date]['open_interest'] = point['y']
                    else:
                        date_map[date][series_name.lower().replace(' ', '_')] = point['y']

            all_records.extend(date_map.values())

        if not all_records:
            return None

        df = pd.DataFrame(all_records)

        if 'date' in df.columns:
            df = df.sort_values('date')
            df['date'] = pd.to_datetime(df['date'])

        return df

    def close(self):
        if self.driver:
            self.driver.quit()

print("CryptoQuantScraper 클래스가 로드되었습니다.")

## 4. 데이터 추출 실행

In [ ]:
# CryptoQuant URL
url = "https://cryptoquant.com/asset/btc/chart/derivatives/open-interest?exchange=all_exchange&symbol=all_symbol&window=DAY&sma=0&ema=0&priceScale=log&metricScale=linear&chartStyle=line"

# 스크래퍼 초기화
scraper = CryptoQuantScraper(headless=True)

try:
    # 데이터 추출
    df = scraper.extract_highcharts_data(url)
    
    if df is not None and not df.empty:
        print(f"\n총 {len(df)}개의 데이터 포인트를 추출했습니다.")
        print(f"\n컬럼: {list(df.columns)}")
        print("\n처음 10개 데이터:")
        display(df.head(10))
        print("\n마지막 10개 데이터:")
        display(df.tail(10))
    else:
        print("데이터를 추출하지 못했습니다.")
        
finally:
    scraper.close()

## 5. 데이터 저장

In [ ]:
# CSV 파일로 저장
if df is not None and not df.empty:
    filename = 'cryptoquant_btc_data.csv'
    df.to_csv(filename, index=False)
    print(f"데이터가 '{filename}' 파일로 저장되었습니다.")
    
    # 파일 다운로드 (Colab에서)
    from google.colab import files
    files.download(filename)

## 6. 데이터 분석 및 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if df is not None and not df.empty:
    # 날짜별 정렬
    df_sorted = df.sort_values('date')
    
    # 그래프 생성
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # BTC 가격 그래프
    if 'price_usd' in df_sorted.columns:
        ax1.plot(df_sorted['date'], df_sorted['price_usd'], color='blue', linewidth=2)
        ax1.set_title('BTC Price (USD)', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Price (USD)', fontsize=12)
        ax1.grid(True, alpha=0.3)
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 미결제 약정 그래프
    if 'open_interest' in df_sorted.columns:
        ax2.plot(df_sorted['date'], df_sorted['open_interest'], color='purple', linewidth=2)
        ax2.set_title('Open Interest', fontsize=14, fontweight='bold')
        ax2.set_ylabel('Open Interest', fontsize=12)
        ax2.set_xlabel('Date', fontsize=12)
        ax2.grid(True, alpha=0.3)
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    # 기본 통계
    print("\n데이터 통계:")
    print(df_sorted.describe())

## 7. 특정 날짜 데이터 조회

In [ ]:
# 특정 날짜의 데이터 조회 예시
if df is not None and not df.empty:
    # 가장 최근 데이터
    latest = df.loc[df['date'].idxmax()]
    print("가장 최근 데이터:")
    print(f"날짜: {latest['date']}")
    if 'price_usd' in latest:
        print(f"BTC 가격: ${latest['price_usd']:,.2f}")
    if 'open_interest' in latest:
        print(f"미결제 약정: ${latest['open_interest']:,.2f}")
    
    # 특정 기간 필터링 예시
    # df_filtered = df[df['date'] >= '2025-01-01']
    # print(f"\n2025년 이후 데이터: {len(df_filtered)}개")